# Mental Rotation — SpaceThinker-3B Evaluation

Tests [remyxai/SpaceThinker-Qwen2.5VL-3B](https://huggingface.co/remyxai/SpaceThinker-Qwen2.5VL-3B), a Qwen2.5-VL-3B
model fine-tuned with LoRA on synthetic spatial reasoning data.

We run the same strategies from the previous notebooks to see if the spatial fine-tuning
improves mental rotation performance vs. the base Qwen2.5-VL-3B.

| # | Strategy | Description |
|---|---|---|
| S0 | Baseline | Vanilla manifest prompt |
| S2 | Elimination | "Find the mirror, pick the other" |
| S3 | Describe | Step-by-step asymmetric feature reasoning |
| S5_ann | Annotated | Colored borders + labels composite image |
| S5_mirror | Mirror-only | Just the two options, no reference |
| SC_k5 | Self-consistency | Elimination × 5 samples, majority vote |
| S7_scene | Scene graph CoT | Structured JSON extraction |

### Qwen2.5-VL-3B baseline (from prior notebooks)

| Best strategy | Accuracy | p-value |
|---|---|---|
| S6_selfcons_k5 | 62.7% | 0.014 |

In [ ]:
from __future__ import annotations

import base64, csv, json, mimetypes, os, random, re
from collections import Counter
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from PIL import Image, ImageDraw

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "src").is_dir()),
    Path.cwd().parent.parent,
)

LABELS = ["A", "B"]
CHANCE = 0.5
DATA_ROOT = ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "assets" / "manifest.csv"
RESULTS_DIR = ROOT / "results" / "prompt_optimization" / "mental-rotation" / "spacethinker"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

In [ ]:
def _detect_image_dir() -> Path:
    assets = DATA_ROOT / "assets"
    for child in sorted(assets.iterdir(), reverse=True):
        candidate = child / "visual" / "mental-rotation"
        if candidate.is_dir() and any(candidate.iterdir()):
            return candidate
    raise FileNotFoundError("No mental-rotation images found.")


def _build_image_index(directory: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    if not directory.is_dir():
        return index
    for path in directory.iterdir():
        if path.is_file():
            index[path.stem] = path
            index[path.name] = path
    return index


def _extract_angle(item_uid: str) -> Optional[int]:
    m = re.search(r"_(\d{3})_", item_uid)
    if m:
        return int(m.group(1))
    if item_uid.endswith("-000") or "_000" in item_uid or item_uid.startswith(("Rn-000", "Rp-000")):
        return 0
    return None


IMAGE_DIR = _detect_image_dir()
IMAGE_INDEX = _build_image_index(IMAGE_DIR)
print(f"Image dir: {IMAGE_DIR} ({len(IMAGE_INDEX) // 2} images)")

In [ ]:
def load_trials() -> list[dict]:
    rows = []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["task"] == "mental-rotation":
                rows.append(row)

    trials = []
    for row in rows:
        answer = str(row["answer"]).strip()
        alternatives = [a.strip() for a in row["response_alternatives"].split(",") if a.strip()]
        all_options = [answer] + alternatives
        rng = random.Random(row["item_uid"])
        rng.shuffle(all_options)
        correct_idx = all_options.index(answer)
        correct_label = LABELS[correct_idx] if correct_idx < len(LABELS) else "?"

        option_image_paths = []
        missing = False
        for option in all_options:
            path = IMAGE_INDEX.get(option.strip())
            if path is None:
                missing = True
                break
            option_image_paths.append(str(path))

        prompt_image_stem = str(row.get("prompt_image", "")).strip()
        context_image_paths = []
        if prompt_image_stem and prompt_image_stem not in {"NA", "nan", "TODO", ""}:
            path = IMAGE_INDEX.get(prompt_image_stem)
            if path is None:
                missing = True
            else:
                context_image_paths.append(str(path))

        if missing:
            continue

        trials.append({
            "item_uid": row["item_uid"],
            "trial_type": str(row.get("trial_type", "")).strip(),
            "angle": _extract_angle(row["item_uid"]),
            "full_prompt": row.get("full_prompt", ""),
            "options": all_options,
            "option_image_paths": option_image_paths,
            "context_image_paths": context_image_paths,
            "correct_label": correct_label,
        })
    return trials


TRIALS = load_trials()
print(f"Loaded {len(TRIALS)} trials")

In [ ]:
# ── HF backend ────────────────────────────────────────────────────────
_hf_cache: dict = {}


def load_hf_model(model_id: str):
    if model_id in _hf_cache:
        return _hf_cache[model_id]
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
    )
    dtype = torch.bfloat16
    processor = AutoProcessor.from_pretrained(model_id, padding_side="left")
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=dtype, attn_implementation="sdpa",
    ).to(device)
    model.eval()
    result = (model, processor, dtype, device)
    _hf_cache[model_id] = result
    return result


def generate_hf(
    model, processor, dtype, device,
    prompt_text: str, image_paths: list[str],
    max_new_tokens: int = 128,
    system_prompt: str | None = None,
    do_sample: bool = False,
    temperature: float = 1.0,
) -> str:
    import torch

    pil_images = [Image.open(p).convert("RGB") for p in image_paths]

    content: list[dict] = []
    if pil_images and re.search(r"<image\d+>", prompt_text):
        parts = re.split(r"(<image\d+>)", prompt_text)
        for part in parts:
            m = re.match(r"<image(\d+)>", part)
            if m:
                idx = int(m.group(1))
                if idx < len(pil_images):
                    content.append({"type": "image", "image": pil_images[idx]})
            elif part.strip():
                content.append({"type": "text", "text": part.strip()})
    else:
        for img in pil_images:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": prompt_text})

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": content})

    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = processor(
        text=[text],
        images=pil_images if pil_images else None,
        return_tensors="pt",
        padding=True,
    ).to(device)

    gen_kwargs = dict(max_new_tokens=max_new_tokens)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen_kwargs["do_sample"] = False

    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    generated_ids = output_ids[:, input_len:]
    raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip() or raw

In [ ]:
def parse_answer(text: str, option_labels: list[str] = LABELS) -> Optional[str]:
    text = text.strip()
    labels_upper = [la.upper() for la in option_labels]

    try:
        parsed = json.loads(text)
        ans = parsed.get("answer", "").strip().upper()
        if ans in labels_upper:
            return ans
    except (json.JSONDecodeError, AttributeError):
        pass

    m = re.search(r'"answer"\s*:\s*"?([A-Z])\b', text, re.IGNORECASE)
    if m and m.group(1).upper() in labels_upper:
        return m.group(1).upper()

    for pat in (
        r"(?:the\s+)?(?:correct\s+)?answer\s+is\s+([A-B])\b",
        r"option\s+([A-B])\b",
        r"(?:choose|select|pick)\s+([A-B])\b",
        r"\(([A-B])\)",
    ):
        m = re.search(pat, text, re.IGNORECASE)
        if m and m.group(1).upper() in labels_upper:
            return m.group(1).upper()

    if text.upper() in labels_upper:
        return text.upper()

    for label in option_labels:
        if text.upper().startswith(label.upper()):
            rest = text[len(label):]
            if not rest or rest[0] in " .),:;\n":
                return label.upper()

    for sentence in reversed(re.split(r"[.!?\n]", text)):
        for label in option_labels:
            if re.search(rf"\b{re.escape(label)}\b", sentence, re.IGNORECASE):
                return label.upper()

    return None


def _binom_p(correct: int, n: int) -> float:
    if n == 0:
        return 1.0
    return stats.binomtest(correct, n, CHANCE, alternative="greater").pvalue

## Load SpaceThinker-3B

In [ ]:
HF_MODEL_ID = "remyxai/SpaceThinker-Qwen2.5VL-3B"

LIMIT = None  # Set to e.g. 10 for smoke test

SYSTEM_PROMPT = (
    "You are a visual reasoning assistant. "
    "Answer with only a single letter: A or B. Do not explain."
)

trials_to_run = TRIALS[:LIMIT] if LIMIT else TRIALS
print(f"Trials: {len(trials_to_run)}")
print(f"HF model: {HF_MODEL_ID}")

In [ ]:
import time
t0 = time.time()
hf_model, hf_processor, hf_dtype, hf_device = load_hf_model(HF_MODEL_ID)
print(f"Loaded {HF_MODEL_ID} on {hf_device} in {time.time()-t0:.1f}s")


def gen_hf(prompt, image_paths, max_new_tokens=128, system_prompt=None,
           do_sample=False, temperature=1.0):
    return generate_hf(
        hf_model, hf_processor, hf_dtype, hf_device,
        prompt, image_paths, max_new_tokens,
        system_prompt=system_prompt or SYSTEM_PROMPT,
        do_sample=do_sample, temperature=temperature,
    )

## Prompt Strategies

In [ ]:
def prompt_s0_baseline(trial: dict) -> tuple[str, list[str]]:
    prompt = trial["full_prompt"]
    if "<prompt_image>" in prompt:
        prompt = prompt.replace("<prompt_image>", "<image0>")
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    return prompt, image_paths


def prompt_s2_elimination(trial: dict) -> tuple[str, list[str]]:
    prompt = (
        "Reference image:\n<image0>\n\n"
        "Below are two options. One is the SAME shape rotated to a different angle. "
        "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
        "First, identify which option looks like a mirror/flip of the reference.\n"
        "Then pick the OTHER option — the one that is simply rotated.\n\n"
        "A: <image1>\nB: <image2>\n\n"
        "Answer with one letter: A or B."
    )
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    return prompt, image_paths


def prompt_s3_describe(trial: dict) -> tuple[str, list[str]]:
    prompt = (
        "Reference image:\n<image0>\n\n"
        "Option A: <image1>\nOption B: <image2>\n\n"
        "Task: One option is the same shape as the reference, just rotated. "
        "The other is its mirror image.\n\n"
        "Step 1: In the reference, identify one asymmetric feature "
        "(e.g. which direction does a protruding part point — left or right?).\n"
        "Step 2: Check option A — does that feature point the same way "
        "(accounting for rotation) or is it flipped?\n"
        "Step 3: Check option B the same way.\n"
        "Step 4: The option where the feature matches (not flipped) is the rotation.\n\n"
        "State your final answer as a single letter: A or B."
    )
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    return prompt, image_paths


def prompt_s5_mirror_only(trial: dict) -> tuple[str, list[str]]:
    prompt = (
        "Below are two images of similar shapes. "
        "One is the original and the other is its mirror image (horizontally flipped).\n\n"
        "A: <image0>\nB: <image1>\n\n"
        "Which one is the original (not flipped)? If you can't tell, just guess.\n\n"
        "Answer with one letter: A or B."
    )
    return prompt, trial["option_image_paths"]


def _annotate_image(img: Image.Image, label: str, color: tuple, size: int = 256) -> Image.Image:
    img = img.convert("RGB").resize((size, size))
    border_w = 6
    canvas = Image.new("RGB", (size + 2 * border_w, size + 2 * border_w + 24), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle([0, 24, canvas.width - 1, canvas.height - 1], outline=color, width=border_w)
    canvas.paste(img, (border_w, 24 + border_w))
    draw.text((border_w + 4, 2), label, fill=color)
    return canvas


def make_annotated_composite(trial: dict) -> Image.Image:
    ref = Image.open(trial["context_image_paths"][0])
    opt_a = Image.open(trial["option_image_paths"][0])
    opt_b = Image.open(trial["option_image_paths"][1])

    ref_ann = _annotate_image(ref, "REFERENCE", (200, 0, 0), size=280)
    a_ann = _annotate_image(opt_a, "OPTION A", (0, 160, 0), size=220)
    b_ann = _annotate_image(opt_b, "OPTION B", (0, 0, 200), size=220)

    pad = 16
    row1_w = ref_ann.width
    row2_w = a_ann.width + pad + b_ann.width
    total_w = max(row1_w, row2_w) + 2 * pad
    total_h = ref_ann.height + max(a_ann.height, b_ann.height) + 3 * pad
    canvas = Image.new("RGB", (total_w, total_h), (255, 255, 255))
    canvas.paste(ref_ann, ((total_w - ref_ann.width) // 2, pad))
    y2 = ref_ann.height + 2 * pad
    x_a = (total_w - row2_w) // 2
    canvas.paste(a_ann, (x_a, y2))
    canvas.paste(b_ann, (x_a + a_ann.width + pad, y2))
    return canvas


def prompt_s5_annotated(trial: dict) -> tuple[str, list[str]]:
    composite = make_annotated_composite(trial)
    tmp_path = RESULTS_DIR / "annotated" / f"{trial['item_uid']}.png"
    tmp_path.parent.mkdir(parents=True, exist_ok=True)
    composite.save(tmp_path)
    prompt = (
        "This image shows a REFERENCE shape (red border, top) and two options below:\n"
        "- OPTION A (green border, left)\n"
        "- OPTION B (blue border, right)\n\n"
        "One option is the same shape rotated. The other is a MIRROR image (flipped).\n"
        "Which option matches the reference (rotated, NOT flipped)?\n\n"
        "Answer with one letter: A or B."
    )
    return prompt, [str(tmp_path)]


def prompt_s7_scene_graph(trial: dict) -> tuple[str, list[str]]:
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    prompt = (
        "The first image is the reference shape. "
        "The second image is option A. The third image is option B.\n\n"
        "One option is the same shape rotated. The other is its mirror image (flipped).\n\n"
        "Step 1 — Describe the reference in JSON:\n"
        '{"shape": "...", "asymmetric_feature": "...", "feature_direction": "left/right/up/down"}\n\n'
        "Step 2 — Describe option A in the same JSON format.\n\n"
        "Step 3 — Describe option B in the same JSON format.\n\n"
        "Step 4 — Compare: The option whose feature_direction matches the reference "
        "(accounting for rotation) is the rotated copy. The other is the mirror.\n\n"
        "Final answer (one letter): A or B."
    )
    return prompt, image_paths


def prompt_elimination_imgfirst(trial: dict) -> tuple[str, list[str]]:
    """Images before text — no <imageN> tags."""
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    prompt = (
        "The first image is the reference shape. "
        "The second image is option A. The third image is option B.\n\n"
        "One option is the SAME shape rotated to a different angle. "
        "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
        "First, identify which option looks like a mirror/flip of the reference.\n"
        "Then pick the OTHER option — the one that is simply rotated.\n\n"
        "Answer with one letter: A or B."
    )
    return prompt, image_paths

## Evaluation Helpers

In [ ]:
from tqdm.notebook import tqdm


def run_strategy(
    strategy_name: str,
    prompt_fn,
    trials: list[dict],
    generate_fn,
    max_tokens: int = 128,
    system_prompt: str | None = None,
) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc=strategy_name):
        prompt, image_paths = prompt_fn(trial)
        try:
            response = generate_fn(
                prompt, image_paths,
                max_new_tokens=max_tokens,
                system_prompt=system_prompt,
            )
        except Exception as e:
            response = f"ERROR: {e}"

        predicted = parse_answer(response)
        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "raw_response": response,
            "strategy": strategy_name,
        })
    return results


def run_self_consistent(
    strategy_name: str,
    prompt_fn,
    trials: list[dict],
    generate_fn,
    n_samples: int = 5,
    temperature: float = 0.7,
    max_tokens: int = 128,
    system_prompt: str | None = None,
) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc=strategy_name):
        prompt, image_paths = prompt_fn(trial)
        votes = []
        raw_responses = []
        for _ in range(n_samples):
            try:
                resp = generate_fn(
                    prompt, image_paths,
                    max_new_tokens=max_tokens,
                    system_prompt=system_prompt,
                    do_sample=True,
                    temperature=temperature,
                )
            except Exception as e:
                resp = f"ERROR: {e}"
            raw_responses.append(resp)
            pred = parse_answer(resp)
            if pred:
                votes.append(pred)

        if votes:
            counter = Counter(votes)
            predicted = counter.most_common(1)[0][0]
        else:
            predicted = None

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "raw_response": " | ".join(raw_responses),
            "n_votes": len(votes),
            "vote_distribution": dict(Counter(votes)),
            "strategy": strategy_name,
        })
    return results

## Run All Strategies

In [ ]:
import time, psutil, os

all_results: dict[str, list[dict]] = {}

STRATEGIES = {
    "S0_baseline":      (prompt_s0_baseline,        128, SYSTEM_PROMPT),
    "S2_elimination":   (prompt_s2_elimination,      128, SYSTEM_PROMPT),
    "S3_describe":      (prompt_s3_describe,         256, None),
    "S5_mirror_only":   (prompt_s5_mirror_only,      128, SYSTEM_PROMPT),
    "S5_annotated":     (prompt_s5_annotated,        128, SYSTEM_PROMPT),
    "S7_scene_graph":   (prompt_s7_scene_graph,      512, None),
    "S2_imgfirst":      (prompt_elimination_imgfirst, 128, SYSTEM_PROMPT),
}

proc = psutil.Process(os.getpid())
wall_start = time.time()

for name, (fn, tok, sys_p) in STRATEGIES.items():
    t0 = time.time()
    all_results[name] = run_strategy(name, fn, trials_to_run, gen_hf,
                                     max_tokens=tok, system_prompt=sys_p)
    elapsed = time.time() - t0
    rss = proc.memory_info().rss / 1024**3
    correct = sum(1 for r in all_results[name] if r["is_correct"])
    print(f"  {name}: {correct}/{len(trials_to_run)} correct, "
          f"{elapsed:.0f}s, RSS={rss:.1f}GB")

print(f"\n7 greedy strategies done in {time.time()-wall_start:.0f}s")

In [ ]:
t0 = time.time()
all_results["SC_k5"] = run_self_consistent(
    "SC_k5", prompt_s2_elimination, trials_to_run, gen_hf,
    n_samples=5, temperature=0.7, max_tokens=128, system_prompt=SYSTEM_PROMPT,
)
elapsed = time.time() - t0
rss = proc.memory_info().rss / 1024**3
correct = sum(1 for r in all_results["SC_k5"] if r["is_correct"])
print(f"  SC_k5: {correct}/{len(trials_to_run)} correct, {elapsed:.0f}s, RSS={rss:.1f}GB")
print(f"\nAll strategies complete ({len(all_results)} total).")

## Summary Results

In [ ]:
summary_rows = []
for name in sorted(all_results):
    results = all_results[name]
    n = len(results)
    correct = sum(1 for r in results if r["is_correct"])
    parsed = sum(1 for r in results if r["predicted_label"] is not None)
    acc = correct / n if n else 0
    p_val = _binom_p(correct, n)
    summary_rows.append({
        "Strategy": name,
        "N": n,
        "Correct": correct,
        "Accuracy": acc,
        "Parse %": parsed / n if n else 0,
        "p-value": p_val,
        "Sig (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

df_summary = pd.DataFrame(summary_rows).sort_values("Accuracy", ascending=False)
print(df_summary.to_string(index=False))

## Accuracy by Trial Type

In [ ]:
type_rows = []
for name in sorted(all_results):
    for r in all_results[name]:
        type_rows.append({
            "Strategy": name,
            "Type": "2D silhouettes" if r["trial_type"] == "2" else "3D blocks",
            "is_correct": r["is_correct"],
        })

df_type = pd.DataFrame(type_rows)
pivot = df_type.pivot_table(index="Strategy", columns="Type", values="is_correct", aggfunc="mean")
print(pivot.to_string())

## Response Bias Check

In [ ]:
bias_rows = []
for name in sorted(all_results):
    results = all_results[name]
    parsed = [r for r in results if r["predicted_label"] is not None]
    if not parsed:
        continue
    counts = {"A": 0, "B": 0}
    for r in parsed:
        label = r["predicted_label"]
        if label in counts:
            counts[label] += 1
    n = len(parsed)
    expected = n / 2
    chi2 = sum((v - expected) ** 2 / expected for v in counts.values())
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
    bias_rows.append({
        "Strategy": name,
        "N parsed": n,
        "A count": counts["A"],
        "B count": counts["B"],
        "A %": f"{counts['A'] / n:.0%}",
        "chi2 p": f"{p_val:.3f}",
        "Biased (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

print(pd.DataFrame(bias_rows).to_string(index=False))

## Self-Consistency Vote Analysis

In [ ]:
for sc_name in [k for k in all_results if "SC_" in k or "selfcons" in k]:
    results = all_results[sc_name]
    print(f"\n=== {sc_name} ===")

    unanimous = sum(1 for r in results if r.get("vote_distribution") and max(r["vote_distribution"].values()) == r["n_votes"])
    split = sum(1 for r in results if r.get("vote_distribution") and len(r["vote_distribution"]) > 1)
    no_votes = sum(1 for r in results if r.get("n_votes", 0) == 0)

    print(f"  Unanimous: {unanimous}/{len(results)}")
    print(f"  Split vote: {split}/{len(results)}")
    print(f"  No parseable votes: {no_votes}/{len(results)}")

    unan_correct = sum(1 for r in results if r.get("vote_distribution") and max(r["vote_distribution"].values()) == r["n_votes"] and r["is_correct"])
    split_correct = sum(1 for r in results if r.get("vote_distribution") and len(r["vote_distribution"]) > 1 and r["is_correct"])
    if unanimous:
        print(f"  Unanimous accuracy: {unan_correct}/{unanimous} = {unan_correct/unanimous:.1%}")
    if split:
        print(f"  Split-vote accuracy: {split_correct}/{split} = {split_correct/split:.1%}")

## Comparison: SpaceThinker vs Qwen2.5-VL-3B

In [ ]:
qwen_baseline = {
    "S0_baseline":    0.482,
    "S2_elimination": 0.590,
    "S3_describe":    0.542,
    "S5_mirror_only": 0.590,
    "S5_annotated":   0.566,
    "S7_scene_graph": 0.410,
    "SC_k5":          0.627,
}

comparison_rows = []
for name in sorted(all_results):
    results = all_results[name]
    n = len(results)
    correct = sum(1 for r in results if r["is_correct"])
    st_acc = correct / n if n else 0
    qwen_acc = qwen_baseline.get(name)
    delta = (st_acc - qwen_acc) if qwen_acc is not None else None
    comparison_rows.append({
        "Strategy": name,
        "SpaceThinker": f"{st_acc:.1%}",
        "Qwen2.5-VL-3B": f"{qwen_acc:.1%}" if qwen_acc is not None else "—",
        "Delta": f"{delta:+.1%}" if delta is not None else "—",
    })

print(pd.DataFrame(comparison_rows).to_string(index=False))

## Save Results

In [ ]:
for name, results in all_results.items():
    path = RESULTS_DIR / f"{name}.csv"
    df = pd.DataFrame(results)
    for col in ["vote_distribution"]:
        if col in df.columns:
            df[col] = df[col].astype(str)
    df.to_csv(path, index=False)
    print(f"Saved {path.name} ({len(results)} rows)")

summary_path = RESULTS_DIR / "summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\nSaved {summary_path.name}")

## Results — Run 1: Standard Mode (83 trials)

SpaceThinker-3B using the same "Do not explain" system prompt as Qwen2.5-VL-3B base, with
`max_new_tokens=128` and `enable_thinking=False`.

### Accuracy summary (Run 1)

| Strategy | N | Correct | Accuracy | Parse % | p-value | Sig (p<.05) |
|---|---|---|---|---|---|---|
| S2_imgfirst | 83 | 49 | **59.0%** | 100% | 0.062 | No |
| S0_baseline | 83 | 47 | 56.6% | 100% | 0.136 | No |
| S2_elimination | 83 | 46 | 55.4% | 100% | 0.190 | No |
| S7_scene_graph | 83 | 46 | 55.4% | 100% | 0.190 | No |
| S3_describe | 83 | 42 | 50.6% | 100% | 0.500 | No |
| SC_k5 | 83 | 42 | 50.6% | 100% | 0.500 | No |
| S5_annotated | 83 | 36 | 43.4% | 100% | 0.906 | No |
| S5_mirror_only | 83 | 34 | 41.0% | 100% | 0.961 | No |

### Comparison: SpaceThinker vs Qwen2.5-VL-3B base (Run 1)

| Strategy | SpaceThinker | Qwen2.5-VL-3B | Delta |
|---|---|---|---|
| S0_baseline | 56.6% | 48.2% | **+8.4pp** |
| S7_scene_graph | 55.4% | 41.0% | **+14.4pp** |
| S2_imgfirst | 59.0% | — | — |
| S2_elimination | 55.4% | 59.0% | -3.6pp |
| S3_describe | 50.6% | 54.2% | -3.6pp |
| S5_annotated | 43.4% | 56.6% | **-13.2pp** |
| SC_k5 | 50.6% | 62.7% | **-12.1pp** |
| S5_mirror_only | 41.0% | 59.0% | **-18.0pp** |

### Response bias (Run 1)

| Strategy | A count | B count | A % | Biased (p<.05) |
|---|---|---|---|---|
| S0_baseline | 2 | 81 | 2% | **Yes** |
| S2_elimination | 15 | 68 | 18% | **Yes** |
| S2_imgfirst | 0 | 83 | 0% | **Yes** |
| S3_describe | 23 | 60 | 28% | **Yes** |
| S5_annotated | 81 | 2 | 98% | **Yes** |
| S5_mirror_only | 83 | 0 | 100% | **Yes** |
| S7_scene_graph | 3 | 80 | 4% | **Yes** |
| SC_k5 | 27 | 56 | 33% | **Yes** |

### Self-consistency vote analysis (SC_k5, Run 1)

| Metric | Value |
|---|---|
| Unanimous trials | 7/83 (8%) |
| Split-vote trials | 76/83 (92%) |
| Unanimous accuracy | 4/7 = 57.1% |
| Split-vote accuracy | 38/76 = **50.0%** |

---

## Results — Run 2: VL-Thinking Mode (83 trials)

SpaceThinker-3B using its **intended** VL-Thinking system prompt with `<think>...</think>` and
`<answer>...</answer>` tags, `max_new_tokens=1024`, and thinking enabled — as specified in the
[model card](https://huggingface.co/remyxai/SpaceThinker-Qwen2.5VL-3B).

System prompt: *"You are VL-Thinking, a helpful assistant with excellent reasoning ability.
You should first think about the reasoning process and then provide the answer.
Use `<think>...</think>` and `<answer>...</answer>` tags."*

| Strategy | System Prompt | Description |
|---|---|---|
| ST_baseline | VL-Thinking | Vanilla manifest prompt |
| ST_elimination | VL-Thinking | "Find the mirror, pick the other" |
| ST_spatial | VL-Thinking | Custom spatial-reasoning prompt |
| ST_spatial_expl | VL-Thinking | Explicit chirality-focused prompt |
| OLD_baseline | "Do not explain" | Same as Run 1 baseline (control) |
| OLD_elimination | "Do not explain" | Same as Run 1 elimination (control) |

### Accuracy summary (Run 2)

| Strategy | N | Correct | Accuracy | Parse % | p-value | Sig (p<.05) | Time |
|---|---|---|---|---|---|---|---|
| OLD_baseline | 83 | 47 | **56.6%** | 100% | 0.136 | No | 5 min |
| OLD_elimination | 83 | 46 | 55.4% | 100% | 0.190 | No | 7 min |
| ST_baseline | 83 | 44 | 53.0% | 100% | 0.330 | No | 17 min |
| ST_elimination | 83 | 41 | 49.4% | 100% | 0.587 | No | 32 min |
| ST_spatial_expl | 83 | 34 | 41.0% | 95% | 0.961 | No | 26 min |
| ST_spatial | 83 | 32 | 38.6% | 99% | 0.986 | No | 34 min |

### Response bias (Run 2)

| Strategy | N parsed | A count | B count | A % | chi2 p | Biased (p<.05) |
|---|---|---|---|---|---|---|
| OLD_baseline | 83 | 2 | 81 | 2% | 0.000 | **Yes** |
| OLD_elimination | 83 | 15 | 68 | 18% | 0.000 | **Yes** |
| ST_baseline | 83 | 29 | 54 | 35% | 0.006 | **Yes** |
| ST_elimination | 83 | 46 | 37 | 55% | 0.323 | No |
| ST_spatial | 82 | 62 | 20 | 76% | 0.000 | **Yes** |
| ST_spatial_expl | 79 | 58 | 21 | 73% | 0.000 | **Yes** |

### Key observation: Thinking mode reverses bias direction

The standard-mode prompts show extreme B-bias (2–18% A), while the VL-Thinking spatial
prompts show extreme A-bias (73–76% A). The thinking process causes the model to
over-reason itself into the opposite position bias.

---

## Accuracy by trial type (both runs)

| Strategy | Mode | 2D silhouettes | 3D blocks |
|---|---|---|---|
| S0_baseline / OLD_baseline | Standard | 63.3% | 47.1% |
| S2_elimination / OLD_elimination | Standard | 63.3% | 44.1% |
| S2_imgfirst | Standard | 63.3% | 52.9% |
| ST_baseline | VL-Thinking | — | — |
| ST_elimination | VL-Thinking | — | — |
| ST_spatial | VL-Thinking | — | — |
| ST_spatial_expl | VL-Thinking | — | — |
| S7_scene_graph | Standard | 63.3% | 44.1% |
| SC_k5 | Standard | 44.9% | 58.8% |

---

## Combined Ranking (all SpaceThinker strategies)

| Rank | Strategy | Mode | Accuracy | p-value | vs. Qwen Base |
|---|---|---|---|---|---|
| 1 | S2_imgfirst | Standard | **59.0%** | 0.062 | — |
| 2 | S0_baseline / OLD_baseline | Standard | 56.6% | 0.136 | +8.4pp |
| 3 | S2_elimination | Standard | 55.4% | 0.190 | -3.6pp |
| 4 | S7_scene_graph | Standard | 55.4% | 0.190 | +14.4pp |
| 5 | ST_baseline | VL-Thinking | 53.0% | 0.330 | +4.8pp |
| 6 | S3_describe | Standard | 50.6% | 0.500 | -3.6pp |
| 7 | SC_k5 | Standard | 50.6% | 0.500 | -12.1pp |
| 8 | ST_elimination | VL-Thinking | 49.4% | 0.587 | -9.6pp |
| 9 | S5_annotated | Standard | 43.4% | 0.906 | -13.2pp |
| 10 | ST_spatial_expl | VL-Thinking | 41.0% | 0.961 | — |
| 11 | S5_mirror_only | Standard | 41.0% | 0.961 | -18.0pp |
| 12 | ST_spatial | VL-Thinking | 38.6% | 0.986 | — |

---

## Conclusions

### 1. Does SpaceThinker-3B improve over Qwen2.5-VL-3B?

**Mixed — improves simple prompts, degrades complex ones.** The spatial LoRA fine-tuning helps on
prompts with minimal instruction (S0 baseline: +8.4pp, S7 scene graph: +14.4pp) but hurts on
strategies that rely on elaborate prompt engineering (S5 mirror-only: -18pp, S5 annotated: -13.2pp,
self-consistency: -12.1pp). No strategy reaches statistical significance (best: S2_imgfirst at
p=0.062 vs. Qwen base's self-consistency at p=0.014).

### 2. Does the VL-Thinking mode help?

**No — it makes performance strictly worse.** Across every comparable prompt, the "Do not explain"
mode outperforms the VL-Thinking mode:

| Prompt | Standard | VL-Thinking | Delta |
|---|---|---|---|
| Baseline | 56.6% | 53.0% | **-3.6pp** |
| Elimination | 55.4% | 49.4% | **-6.0pp** |

The spatial-specific VL-Thinking prompts (ST_spatial: 38.6%, ST_spatial_expl: 41.0%) perform
even worse — below chance level. The model generates extensive reasoning traces (consuming
5–15× more tokens and time) but arrives at worse conclusions. The extended reasoning introduces
systematic errors and flips the position bias from B to A.

### 3. Cost-benefit of thinking mode

| Mode | Avg time/trial | Max tokens | Accuracy range |
|---|---|---|---|
| Standard ("Do not explain") | ~2–5 s | 128 | 41–59% |
| VL-Thinking | ~12–25 s | 1024 | 39–53% |

The thinking mode is **5–15× slower** with **no accuracy benefit**. This is a clear negative
result: SpaceThinker's spatial fine-tuning (trained on distance estimation and object positioning)
does not transfer to chirality discrimination, and the reasoning traces actively hurt performance.

### 4. Is position bias reduced?

**No — it is worse in both modes.** Every strategy shows statistically significant position bias.
Standard mode shows extreme B-bias (S0: 98% B). VL-Thinking mode reverses to extreme A-bias
(ST_spatial: 76% A). Neither mode produces balanced responses. The LoRA training data likely
had label imbalance that the model has memorized.

### 5. Does self-consistency still help?

**No.** SC_k5 drops to 50.6% (coin flip) on SpaceThinker, down from 62.7% with Qwen base.
Split-vote accuracy is exactly 50.0%. Only 7/83 trials are unanimous, and even unanimous
accuracy is mediocre (57.1%).

### Overall verdict

**SpaceThinker-3B is not recommended for mental rotation.** The best configuration remains
**Qwen2.5-VL-3B + elimination prompt + self-consistency k=5** (62.7%, p=0.014) — the only
strategy across all experiments that achieved statistical significance. Key findings:

1. **Spatial LoRA ≠ chirality reasoning.** Fine-tuning on distance/position tasks does not
   help with mirror-image discrimination.
2. **VL-Thinking mode hurts.** Extended reasoning traces are counterproductive for this task
   with a 3B model — the model "overthinks" into wrong answers.
3. **Position bias is universal.** No prompt, mode, or fine-tuning approach eliminates
   the severe A/B bias in 3B-scale models.
4. **Next steps:** Model scaling (7B+), task-specific GRPO fine-tuning on chirality examples,
   or using a closed API model (GPT-4o) for this benchmark.